# `Macro-Hop`: Reinforcement Learning



## load dependencies

In [ ]:
import os
import re
import json
import tempfile

# --------- change these path variables as required
macrohop_dir = os.path.expanduser("/home/hsj/workstations/hoppings/Macro-hop")
macrohop_env = os.path.expanduser("/home/hsj/software/anaconda3/envs/Macro-hop")
output_dir = os.path.expanduser("/home/hsj/workstations/hoppings/Macro-hop/experiment/Macro-hop_PDGFR")

# --------- do not change
# get the notebook's root path
try: ipynb_path
except NameError: ipynb_path = os.getcwd()

# if required, generate a folder to store the results
try:
    os.mkdir(output_dir)
except FileExistsError:
    pass

## initialize the dictionary

In [ ]:
configuration = {
    "version": 3,
    "model_type": "link_invent",
    "run_type": "reinforcement_learning"
}

In [ ]:
configuration["logging"] = {
    "sender": "",                          # only relevant if "recipient" is set to "remote"
    "recipient": "local",                  # either to local logging or use a remote REST-interface
    "logging_path": os.path.join(output_dir, "progress.log"),  # load this folder in tensorboard to 
                                                               # visualize training progress
    "result_folder": os.path.join(output_dir, "results"),      # output directory for results
    "job_name": "Macro-hop RL Demo",     # set an arbitrary job name for identification
    "job_id": "N/A"                        # only relevant if "recipient" is set to "remote"
}

## Add the "parameters" block

In [ ]:
configuration["parameters"] = {}

configuration["parameters"] = {
    "actor": os.path.join(ipynb_path, "macro-hop.ckpt"),   
    "critic": os.path.join(ipynb_path, "macro-hop.ckpt"),
    "warheads" : ["C1(NC2=CC=CC3=C2)=NC=C4C(C(N5CCC(NCCCO3)CC5)=CC=C4)=N1|[*]OCCCNC1CCN([*])CC1"],   #["reference macro|linker"]
    "n_steps": 200,          # number of epochs
    "learning_rate": 0.0001,   # the default value works well but can be increased depending on training progress
    "batch_size": 128,         # the default value works well   [*]F FOCCNC1CCN([*])CC1
    "randomize_warheads": True,
    "learning_strategy": {
        "name": "dap", 
        "parameters": {
        "sigma": 120
        }
    }
}

In [ ]:
configuration["parameters"]["scoring_strategy"] = {
    "name": "link_hop" # do not change
}   

configuration["parameters"]["scoring_strategy"]["diversity_filter"] =  {
      "name": "IdenticalMurckoScaffold"       
}

## Scoring_function

In [ ]:
scoring_function = {
    "name": "custom_product", 
    "parallel": False,  # do not change                   
    # add component: HBD
    "parameters": [
    {
      "weight": 1,
      "component_type": "linker_num_hbd",                 
      "name": "Linker Num HBD",              # this can be arbitrary and is used for user reference
      "specific_parameters": {
      "transformation": {
              "high": 1,
              "low": 0,
              "transformation_type": "step"   #......
              }
           }
      },
    # add component: HBA
    {
      "weight": 1,
      "component_type": "linker_num_hba", 
      "name": "Linker Num HBA",              # this can be arbitrary and is used for user reference
      "specific_parameters": {
      "transformation": {
              "high": 4,
              "low": 2,
              "transformation_type": "step"
              }
           }
      }, 
    # add component: num_aromatic_rings
    {
      "weight": 1,
      "component_type": "linker_num_aromatic_rings", 
      "name": "Linker Num Aromatic Rings",              # this can be arbitrary and is used for user reference
      "specific_parameters": {
      "transformation": {
              "high": 4,
              "low": 2,
              "transformation_type": "step"
              }
           }
      }, 
    # add component: matching_substructure1
    {
        "component_type": "matching_substructure", 
        "name": "Matching substructure1",       # arbitrary name for the component
        "weight": 1,                           
        "specific_parameters": {
            "smiles": ['[#7]:[#6]-[#7H]']    ,       # a match with this substructure is required
        "transformation": {
              "high": 1,
              "low": 1,
              "transformation_type": "step"
              }}
    },
    # add component: MW
    {
        "component_type": "linker_mol_weight", 
        "name": "linker_mol_weight",       # arbitrary name for the component
        "weight": 1,                         
        "specific_parameters": {
        "transformation": {
              "high": 250,
              "low": 200,
              "transformation_type": "step"
              }}
    },
    # add component: linker_custom_alerts
    {
        "component_type": "scaffold_custom_alerts", 
        "name": "linker_custom_alerts",       # arbitrary name for the component
        "weight": 1,                           # the weight of the component (default: 1)
        "specific_parameters": {
        "transformation": {
              "high": 0,
              "low": 0,
              "transformation_type": "step"
              }}
    }, 
    # add component: The largest ring of scaffold
    {
      "weight": 1,
      "component_type": "scaffold_ring_score",  
      "name": "Scaffold Ring Score",              # this can be arbitrary and is used for user reference
      "specific_parameters": {
      "transformation": {
              "high": 8,
              "low": 7,
              "transformation_type": "step"
              }
           }
      },
    # add component: linker_shape_match
    {
        "component_type": "scaffold_rocs_score",    # You need to add reference/reference.pdb at jupyter notebook path
        "name": "linker_rocs_score",       # arbitrary name for the component
        "weight": 0.5,                       # the weight of the component (default: 1)
        "specific_parameters": {
        "transformation": {
              "transformation_type": "no_transformation",
              }}
    }]
}

configuration["parameters"]["scoring_strategy"]["scoring_function"] = scoring_function

In [ ]:
# write out the configuration to disc
configuration_JSON_path = os.path.join(output_dir, "Macro-hop_Configuration.json")
with open(configuration_JSON_path, 'w') as f:
    json.dump(configuration, f, indent=4, sort_keys=False)

# Run


In [ ]:
%%time
%%capture captured_err_stream --no-stderr
# execute REhop from the command-line
!{macrohop_env}/bin/python {macrohop_dir}/input.py {configuration_JSON_path}

In [ ]:
# print the output to a file, just to have it for documentation
with open(os.path.join(output_dir, "run.err"), 'w') as file:
    file.write(captured_err_stream.stdout)

# Analysis

In [ ]:
# import needed packages
import pandas as pd

scaffold_memory_path = os.path.join(output_dir, 'results/scaffold_memory.csv')
df = pd.read_csv(scaffold_memory_path)
df['raw_linker_rocs_score'] = df['raw_linker_rocs_score'].astype(float)
df

In [ ]:
#count_greater_than_1 = len(df[df['raw_linker_rocs_score'] > 1.1] & df[df['total_score'] > 1])
df_score1 = df[(df['total_score'] >= 1.0)]
len(df_score1)

In [ ]:
df_top500 = df_score1.head(1500)
df_top500

In [ ]:
# import needed packages
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw

smile_list = df['SMILES'][0:50].to_list() # change the number here to show more/less top compounds
mols = [Chem.MolFromSmiles(smiles) for smiles in smile_list]
for mol in mols:
    Chem.rdCoordGen.AddCoords(mol)
Draw.MolsToGridImage(mols)